In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,2,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,2,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,2,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,2,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,baseline,0,9,0.0
26996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,baseline,0,9,0.0
26997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,baseline,0,9,0.0
26998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,0,9,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  488251.612663
                                  2                  381162.183392
                                  3                  383451.495271
                                  4                  374358.610912
                                  5                  271585.173914
intervention  maternal_disorders  1                  466030.603811
                                  2                  372212.090198
                                  3                  373272.146054
                                  4                  352623.273189
                                  5                  259808.113310
zero          maternal_disorders  1                  488251.612663
                                  2                  381162.183392
                                  3                  383451.495271
                                  4                  374358.610912
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,2,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,2,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,2,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,2,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
94495,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,0,9,0.0
94496,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,0,9,0.0
94497,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,0,9,0.0
94498,ylds,cause,all_causes,all_causes,95_plus,severe,5,baseline,0,9,0.0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  46768.230034
                                  2                  50529.265527
                                  3                  37324.045990
                                  4                  40581.273762
                                  5                  19654.755110
              maternal_disorders  1                  25228.269140
                                  2                  20675.751455
                                  3                  19650.009535
                                  4                  19641.509110
                                  5                  16017.475491
intervention  anemia              1                  41498.207505
                                  2                  44757.582089
                                  3                  32602.881779
                                  4                  36132.106635
                          

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   46768.230034
                                  2                   50529.265527
                                  3                   37324.045990
                                  4                   40581.273762
                                  5                   19654.755110
              maternal_disorders  1                  513479.881803
                                  2                  401837.934847
                                  3                  403101.504806
                                  4                  394000.120022
                                  5                  287602.649405
intervention  anemia              1                   41498.207505
                                  2                   44757.582089
                                  3                   32602.881779
                                  4                   36132.106635
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # NOTE: This else branch is for processing the Ethiopia results,
    # where no Vivarium sims were run, so the corresponding DALYs should
    # just be 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,334440.482267
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,334471.170619
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,293430.259276
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,269908.450785
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,193603.315485
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,292845.292479
1196,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,266909.197062
1197,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,266120.628846
1198,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,195712.180641


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.491293e+07
                      2                  1.549619e+07
                      3                  1.390872e+07
                      4                  1.230218e+07
                      5                  9.346356e+06
intervention  lbwsg   1                  1.488648e+07
                      2                  1.547270e+07
                      3                  1.387044e+07
                      4                  1.229044e+07
                      5                  9.331674e+06
zero          lbwsg   1                  1.491293e+07
                      2                  1.549619e+07
                      3                  1.390872e+07
                      4                  1.230218e+07
                      5                  9.346356e+06
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,766.877919,zero
1,Female,0.0,0.019178,2,700.282642,zero
2,Female,0.0,0.019178,3,589.931789,zero
3,Female,0.0,0.019178,4,459.736805,zero
4,Female,0.0,0.019178,5,352.172009,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,264.543138,intervention
746,Male,95.0,125.000000,2,232.037383,intervention
747,Male,95.0,125.000000,3,232.795899,intervention
748,Male,95.0,125.000000,4,225.946255,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  293489.769842
                      2                  290634.629142
                      3                  270108.007457
                      4                  261096.090280
                      5                  253149.544942
intervention  anemia  1                  268467.720409
                      2                  263674.136558
                      3                  242633.577006
                      4                  233236.996419
                      5                  225270.242141
zero          anemia  1                  293489.769842
                      2                  290634.629142
                      3                  270108.007457
                      4                  261096.090280
                      5                  253149.544942
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

0.0

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  293493.017208
                      2                  268664.813364
                      3                  197222.364523
                      4                  155609.297594
                      5                  112680.166776
intervention  anemia  1                  269002.467220
                      2                  245338.935847
                      3                  177046.403702
                      4                  138871.094159
                      5                   99284.071491
zero          anemia  1                  293493.017208
                      2                  268664.813364
                      3                  197222.364523
                      4                  155609.297594
                      5                  112680.166776
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

0.0

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  977586.330847
                      2                  918661.844674
                      3                  755336.324359
                      4                  674418.326204
                      5                  587868.028324
intervention  anemia  1                  894525.470953
                      2                  835135.843208
                      3                  677916.774412
                      4                  602373.687838
                      5                  522270.323942
zero          anemia  1                  977586.330847
                      2                  918661.844674
                      3                  755336.324359
                      4                  674418.326204
                      5                  587868.028324
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  412740.875039
                      2                  422053.233994
                      3                  382074.313655
                      4                  338670.157838
                      5                  297460.293500
baseline      ntd     1                  412740.875039
                      2                  422053.233994
                      3                  382074.313655
                      4                  338670.157838
                      5                  297460.293500
intervention  ntd     1                  259207.040176
                      2                  272480.115517
                      3                  269878.685992
                      4                  252438.600444
                      5                  226591.572902
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  1.024355e+06
                                  2                  9.691911e+05
                                  3                  7.926604e+05
                                  4                  7.149996e+05
                                  5                  6.075228e+05
              lbwsg               1                  1.491293e+07
                                  2                  1.549619e+07
                                  3                  1.390872e+07
                                  4                  1.230218e+07
                                  5                  9.346356e+06
              maternal_disorders  1                  5.134799e+05
                                  2                  4.018379e+05
                                  3                  4.031015e+05
                                  4                  3.940001e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)